# Phase 3: Model Calibration Analysis
## DNA Gene Mapping Project
**Author:** Sharique Mohammad  
**Date:** February 2026  

---

## Objective
Evaluate model calibration:
- Check if predicted probabilities match actual outcomes
- Generate calibration (reliability) plots
- Calculate Brier scores
- Assess confidence in predictions

## Why Calibration Matters
A well-calibrated model means:
- 80% probability → 80% chance of being pathogenic
- Probabilities are trustworthy for clinical decisions
- Can set meaningful decision thresholds

---
## 1. Setup

In [1]:
# Imports
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.calibration import calibration_curve, CalibrationDisplay
from sklearn.metrics import brier_score_loss, log_loss

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("OK Imports successful")

OK Imports successful


In [2]:
# Configuration
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data" / "ml"
MODEL_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "data" / "analytical" / "figures" / "phase3"
REPORTS_DIR = PROJECT_ROOT / "data" / "analytical" / "reports"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("="*80)
print("MODEL CALIBRATION ANALYSIS")
print("="*80)

MODEL CALIBRATION ANALYSIS


---
## 2. Load Test Data and Models

In [3]:
print("Loading test datasets...")

# Variant test set
with open(DATA_DIR / "variant_test.pkl", 'rb') as f:
    test_data = pickle.load(f)
    X_test, y_test = test_data['X'], test_data['y']

print(f"Variant test set: {X_test.shape}")

# SV test set
with open(DATA_DIR / "sv_test.pkl", 'rb') as f:
    sv_test = pickle.load(f)
    X_sv_test, y_sv_test = sv_test['X'], sv_test['y']

print(f"SV test set: {X_sv_test.shape}")

Loading test datasets...
Variant test set: (623435, 75)
SV test set: (32512, 13)


In [4]:
print("\nLoading trained models...")

# Load best models
with open(MODEL_DIR / "ensemble_xgb_variants.pkl", 'rb') as f:
    variant_model = pickle.load(f)
print("Loaded variant model (XGBoost)")

with open(MODEL_DIR / "sv_raw_features_best.pkl", 'rb') as f:
    sv_model = pickle.load(f)
print("Loaded SV model (XGBoost)")


Loading trained models...
Loaded variant model (XGBoost)
Loaded SV model (XGBoost)


---
## 3. Variant Model Calibration

In [5]:
print("\n" + "="*80)
print("VARIANT PATHOGENICITY MODEL - CALIBRATION ANALYSIS")
print("="*80)

# Get predicted probabilities
y_proba = variant_model.predict_proba(X_test)[:, 1]

# Calculate calibration curve
prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10, strategy='uniform')

# Calculate Brier score (lower is better, 0 is perfect)
brier_score = brier_score_loss(y_test, y_proba)

# Calculate log loss
logloss = log_loss(y_test, y_proba)

print(f"\nCalibration Metrics:")
print(f"  Brier Score: {brier_score:.4f} (lower is better, 0 is perfect)")
print(f"  Log Loss: {logloss:.4f}")

# Interpret Brier score
if brier_score < 0.1:
    print("  Interpretation: Excellent calibration")
elif brier_score < 0.15:
    print("  Interpretation: Good calibration")
elif brier_score < 0.25:
    print("  Interpretation: Moderate calibration")
else:
    print("  Interpretation: Poor calibration")


VARIANT PATHOGENICITY MODEL - CALIBRATION ANALYSIS

Calibration Metrics:
  Brier Score: 0.0207 (lower is better, 0 is perfect)
  Log Loss: 0.0731
  Interpretation: Excellent calibration


In [6]:
# Plot calibration curve
print("\nGenerating calibration plot...")

fig, ax = plt.subplots(figsize=(10, 8))

# Plot perfectly calibrated line
ax.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration', linewidth=2)

# Plot actual calibration
ax.plot(prob_pred, prob_true, 'o-', label='Variant Model', linewidth=2, markersize=8)

ax.set_xlabel('Predicted Probability', fontsize=12)
ax.set_ylabel('Actual Probability (Fraction Positive)', fontsize=12)
ax.set_title(f'Variant Model Calibration\nBrier Score: {brier_score:.4f}', fontsize=14)
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(FIGURES_DIR / "30_variant_calibration_curve.png", dpi=150, bbox_inches='tight')
plt.close()

print("OK Calibration plot saved")


Generating calibration plot...
OK Calibration plot saved


In [7]:
# Probability distribution analysis
print("\nPredicted Probability Distribution:")

# For actual positives (pathogenic)
proba_positives = y_proba[y_test == True]
print(f"\nPathogenic variants (n={len(proba_positives)}):")
print(f"  Mean probability: {proba_positives.mean():.3f}")
print(f"  Median probability: {np.median(proba_positives):.3f}")
print(f"  Std probability: {proba_positives.std():.3f}")

# For actual negatives (benign)
proba_negatives = y_proba[y_test == False]
print(f"\nBenign variants (n={len(proba_negatives)}):")
print(f"  Mean probability: {proba_negatives.mean():.3f}")
print(f"  Median probability: {np.median(proba_negatives):.3f}")
print(f"  Std probability: {proba_negatives.std():.3f}")


Predicted Probability Distribution:

Pathogenic variants (n=49271):
  Mean probability: 0.855
  Median probability: 0.979
  Std probability: 0.253

Benign variants (n=574164):
  Mean probability: 0.035
  Median probability: 0.001
  Std probability: 0.118


In [8]:
# Plot probability distributions
print("\nGenerating probability distribution plot...")

fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(proba_negatives, bins=50, alpha=0.6, label='Benign', density=True, color='blue')
ax.hist(proba_positives, bins=50, alpha=0.6, label='Pathogenic', density=True, color='red')

ax.axvline(0.5, color='black', linestyle='--', linewidth=2, label='Decision Threshold')
ax.set_xlabel('Predicted Probability', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Predicted Probability Distribution by True Class', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "31_variant_probability_distribution.png", dpi=150, bbox_inches='tight')
plt.close()

print("OK Distribution plot saved")


Generating probability distribution plot...
OK Distribution plot saved


---
## 4. SV Model Calibration

In [10]:
print("\n" + "="*80)
print("SV RISK MODEL - CALIBRATION ANALYSIS")
print("="*80)

# Filter to raw features only (8 features)
raw_features = ['sv_type_class', 'sv_size', 'sv_size_category', 
                'has_gene_overlap', 'affected_gene_count', 'is_multi_gene_sv',
                'affects_pharmacogenes', 'affects_omim_genes']

X_sv_test_raw = X_sv_test[raw_features]

# Get predicted probabilities
y_sv_proba = sv_model.predict_proba(X_sv_test_raw)[:, 1]

# Calculate calibration curve
sv_prob_true, sv_prob_pred = calibration_curve(y_sv_test, y_sv_proba, n_bins=10, strategy='uniform')

# Calculate Brier score
sv_brier_score = brier_score_loss(y_sv_test, y_sv_proba)

# Calculate log loss
sv_logloss = log_loss(y_sv_test, y_sv_proba)

print(f"\nCalibration Metrics:")
print(f"  Brier Score: {sv_brier_score:.4f}")
print(f"  Log Loss: {sv_logloss:.4f}")

if sv_brier_score < 0.1:
    print("  Interpretation: Excellent calibration")
elif sv_brier_score < 0.15:
    print("  Interpretation: Good calibration")
else:
    print("  Interpretation: Moderate calibration")


SV RISK MODEL - CALIBRATION ANALYSIS

Calibration Metrics:
  Brier Score: 0.0154
  Log Loss: 0.0537
  Interpretation: Excellent calibration


In [11]:
# Plot SV calibration curve
print("\nGenerating SV calibration plot...")

fig, ax = plt.subplots(figsize=(10, 8))

ax.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration', linewidth=2)
ax.plot(sv_prob_pred, sv_prob_true, 'o-', label='SV Model', linewidth=2, markersize=8, color='green')

ax.set_xlabel('Predicted Probability', fontsize=12)
ax.set_ylabel('Actual Probability (Fraction Positive)', fontsize=12)
ax.set_title(f'SV Risk Model Calibration\nBrier Score: {sv_brier_score:.4f}', fontsize=14)
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(FIGURES_DIR / "32_sv_calibration_curve.png", dpi=150, bbox_inches='tight')
plt.close()

print("OK SV calibration plot saved")


Generating SV calibration plot...
OK SV calibration plot saved


---
## 5. Confidence Interval Analysis

In [12]:
print("\n" + "="*80)
print("CONFIDENCE INTERVAL ANALYSIS")
print("="*80)

# Bin predictions by confidence level
def analyze_confidence_bins(y_true, y_proba, model_name):
    """
    Analyze performance at different confidence levels
    """
    bins = [
        (0.0, 0.3, 'Low Confidence (0.0-0.3)'),
        (0.3, 0.7, 'Uncertain (0.3-0.7)'),
        (0.7, 1.0, 'High Confidence (0.7-1.0)')
    ]
    
    print(f"\n{model_name} - Performance by Confidence Level:")
    print("="*60)
    
    for low, high, label in bins:
        mask = (y_proba >= low) & (y_proba < high)
        n_samples = mask.sum()
        
        if n_samples == 0:
            continue
        
        y_true_bin = y_true[mask]
        y_proba_bin = y_proba[mask]
        
        # Predictions using 0.5 threshold
        y_pred_bin = (y_proba_bin >= 0.5).astype(int)
        
        accuracy = (y_true_bin == y_pred_bin).mean()
        positive_rate = y_true_bin.mean()
        
        print(f"\n{label}:")
        print(f"  Samples: {n_samples:,} ({n_samples/len(y_true)*100:.1f}%)")
        print(f"  Accuracy: {accuracy:.3f}")
        print(f"  True Positive Rate: {positive_rate:.3f}")
        print(f"  Mean Probability: {y_proba_bin.mean():.3f}")

# Analyze variant model
analyze_confidence_bins(y_test, y_proba, "Variant Model")

# Analyze SV model
analyze_confidence_bins(y_sv_test, y_sv_proba, "SV Model")


CONFIDENCE INTERVAL ANALYSIS

Variant Model - Performance by Confidence Level:

Low Confidence (0.0-0.3):
  Samples: 556,808 (89.3%)
  Accuracy: 0.993
  True Positive Rate: 0.007
  Mean Probability: 0.016

Uncertain (0.3-0.7):
  Samples: 19,937 (3.2%)
  Accuracy: 0.624
  True Positive Rate: 0.242
  Mean Probability: 0.475

High Confidence (0.7-1.0):
  Samples: 46,690 (7.5%)
  Accuracy: 0.874
  True Positive Rate: 0.874
  Mean Probability: 0.946

SV Model - Performance by Confidence Level:

Low Confidence (0.0-0.3):
  Samples: 16,537 (50.9%)
  Accuracy: 0.987
  True Positive Rate: 0.013
  Mean Probability: 0.016

Uncertain (0.3-0.7):
  Samples: 1,073 (3.3%)
  Accuracy: 0.645
  True Positive Rate: 0.564
  Mean Probability: 0.544

High Confidence (0.7-1.0):
  Samples: 14,902 (45.8%)
  Accuracy: 0.995
  True Positive Rate: 0.995
  Mean Probability: 0.990


---
## 6. Save Calibration Results

In [13]:
# Save calibration metrics
calibration_results = {
    'variant_model': {
        'brier_score': float(brier_score),
        'log_loss': float(logloss),
        'mean_prob_positive': float(proba_positives.mean()),
        'mean_prob_negative': float(proba_negatives.mean())
    },
    'sv_model': {
        'brier_score': float(sv_brier_score),
        'log_loss': float(sv_logloss)
    }
}

import json
with open(REPORTS_DIR / "calibration_results.json", 'w') as f:
    json.dump(calibration_results, f, indent=2)

print("\nOK Calibration results saved: calibration_results.json")


OK Calibration results saved: calibration_results.json


---
## 7. Summary and Recommendations

In [14]:
print("\n" + "="*80)
print("CALIBRATION ANALYSIS SUMMARY")
print("="*80)

print("\nVARIANT PATHOGENICITY MODEL:")
print(f"  Brier Score: {brier_score:.4f}")
if brier_score < 0.15:
    print("  Status: Well-calibrated")
    print("  Recommendation: Probabilities can be trusted for decision-making")
else:
    print("  Status: Needs calibration improvement")
    print("  Recommendation: Consider Platt scaling or isotonic regression")

print("\nSV RISK MODEL:")
print(f"  Brier Score: {sv_brier_score:.4f}")
if sv_brier_score < 0.15:
    print("  Status: Well-calibrated")
    print("  Recommendation: Probabilities can be trusted")
else:
    print("  Status: Needs improvement")
    print("  Recommendation: Apply calibration methods")

print("\n" + "="*80)
print("FILES CREATED")
print("="*80)
print("\nFigures:")
print("  - 30_variant_calibration_curve.png")
print("  - 31_variant_probability_distribution.png")
print("  - 32_sv_calibration_curve.png")
print("\nReports:")
print("  - calibration_results.json")

print("\n" + "="*80)
print("CALIBRATION ANALYSIS COMPLETE")
print("="*80)
print("\nKey Takeaways:")
print("  - Calibration tells us if probabilities are trustworthy")
print("  - Lower Brier score = better calibration")
print("  - Well-calibrated models are critical for clinical use")
print("  - If poorly calibrated, apply Platt scaling or isotonic regression")
print("="*80)


CALIBRATION ANALYSIS SUMMARY

VARIANT PATHOGENICITY MODEL:
  Brier Score: 0.0207
  Status: Well-calibrated
  Recommendation: Probabilities can be trusted for decision-making

SV RISK MODEL:
  Brier Score: 0.0154
  Status: Well-calibrated
  Recommendation: Probabilities can be trusted

FILES CREATED

Figures:
  - 30_variant_calibration_curve.png
  - 31_variant_probability_distribution.png
  - 32_sv_calibration_curve.png

Reports:
  - calibration_results.json

CALIBRATION ANALYSIS COMPLETE

Key Takeaways:
  - Calibration tells us if probabilities are trustworthy
  - Lower Brier score = better calibration
  - Well-calibrated models are critical for clinical use
  - If poorly calibrated, apply Platt scaling or isotonic regression
